# Part 5 — Read the existing deployment permissions before retrying

ใช้ WSL/kernel `itcs355-lab3` เดิม รัน code cell เดียว

ผลล่าสุด: ingest / validate / train / evaluate / register ผ่านแล้ว
ล้มเหลวเฉพาะ deploy_staging ด้วย `Startup task failed due to incorrect role assignments on resource`

cell นี้อ่านข้อมูลจริงให้ครบก่อนให้ retry:
- endpoint ใช้ serving identity ที่ตั้งใจไว้หรือไม่
- identity มี Blob data reader บน workspace Storage account และ AcrPull บน ACR หรือไม่
- model version ที่ลงทะเบียนแล้ว พร้อม lineage และตำแหน่ง artifact
- deployment image, MODEL_PATH, inference routes และ storage-initializer/inference logs
- จำนวน compute nodes ณ เวลาตรวจ

**อ่านอย่างเดียวบน Azure: ไม่เปลี่ยน role ไม่ deploy ไม่ submit/train ไม่ commit/push และไม่ลบ resource**
ไม่มีการเปิด compute/endpoint ใหม่; ทรัพยากรที่ยังอยู่ไม่ได้ถูกหยุดโดย cell นี้
เก็บ `reports/deployment-audit.json` แล้วส่ง SEND THIS SUMMARY กลับมา
ยังไม่กดรัน 05 / 05b / 05d ซ้ำ เพื่อเก็บ model version ที่ผ่านแล้วและไม่ฝึกใหม่


In [1]:
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != "Linux" or "microsoft" not in platform.release().lower():
    raise RuntimeError("เลือก kernel itcs355-lab3 ฝั่ง WSL ก่อนครับ")
here = Path.cwd().resolve()
lab = next((base / "lab5" for base in (here, *here.parents)
            if (base / "lab5/scripts/inspect_deployment.py").is_file()
            and (base / "lab4/README.md").is_file()), None)
if lab is None:
    raise RuntimeError("เปิด notebook จาก repository เดิมก่อนครับ")
print("Read existing deployment identity, roles and model only — no deployment retry.", flush=True)
with subprocess.Popen(
    [sys.executable, "-u", "-m", "scripts.inspect_deployment"], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end="", flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        raise
if exit_code:
    raise RuntimeError("อ่านผลเดิมไม่สำเร็จ ส่ง error ด้านบน ไม่ต้องรัน training ใหม่ครับ")
print("ส่ง SEND THIS SUMMARY กลับมา แล้วหยุดตรงนี้ครับ", flush=True)


Read existing deployment identity, roles and model only — no deployment retry.
[16:24:15] 1/4 Read existing pipeline, endpoint, deployment and workspace; no changes
[16:24:31] 2/4 Read serving identity roles at storage account and ACR, including inherited roles
[16:24:35] 3/4 Read the registered candidate's exact version, lineage and artifact path
[16:24:36] 4/4 Read storage-initializer/inference logs and current compute nodes

--- SEND THIS SUMMARY ---
{
  "inspected_at": "2026-10-23T09:24:31.049733+00:00",
  "status": "deployment_audit",
  "job": "lab5-20261023-070538-normal-repair",
  "job_status": "Failed",
  "source_commit": "4f3a8957ded20901989fa48bbe58be29c33f6578",
  "steps": [
    {
      "name": "65857f4b-5bc5-4814-bf5a-d07ec7495591",
      "step": "ingest",
      "status": "Completed"
    },
    {
      "name": "ae0e96a9-64fb-444f-a649-164497e727eb",
      "step": "validate",
      "status": "Completed"
    },
    {
      "name": "f115d4dd-f69c-4195-8c1c-f4a87f9bbf3d",
     